In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

In [ ]:
EOL_PATH = r"C:\EOL_EDA\clean_merged_dataset1.csv"

df = pd.read_csv(EOL_PATH, low_memory=False)
print(f"Raw shape: {df.shape}")

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^a-z0-9_]', '', regex=True)
)
print(f"Columns: {df.columns.tolist()}")

In [ ]:
df = df[~df['no'].astype(str).str.lower().isin(['no', 'nan'])]
df = df[~df['step_name_1'].astype(str).str.lower().str.contains('step name', na=False)]
df.reset_index(drop=True, inplace=True)
print(f"After cleaning headers: {df.shape}")

In [ ]:
df['unit_id'] = df['source_file'].astype(str).str.strip()
print(f"Unique units: {df['unit_id'].nunique()}")

In [ ]:
import pandas as pd

cost_vector = pd.read_csv("eol_cost_vector.csv", index_col=0, header=0)
cost_vector.columns = ['mean_cost_s']

print(f"EOL full sequence cost (sum of mean step times): {cost_vector['mean_cost_s'].sum():.2f} s")
print(f"Number of steps: {len(cost_vector)}")
print(f"\nTop 10 most expensive steps:")
print(cost_vector['mean_cost_s'].sort_values(ascending=False).head(10).to_string())

In [ ]:
df['result'] = df['result'].astype(str).str.strip().str.upper()
df['outcome_from_filename'] = df['outcome_from_filename'].astype(str).str.strip().str.upper()

df['result_binary'] = df['result'].map({'PASS': 1, 'FAIL': 0, 'ABORT': 0})
df['unit_pass']     = df['outcome_from_filename'].map({'PASS': 1, 'FAIL': 0, 'ABORT': 0})

print(f"Result values:  {df['result'].unique()}")
print(f"Unit outcomes:  {df['outcome_from_filename'].unique()}")

In [ ]:
df['step_time']   = pd.to_numeric(df['step_time'], errors='coerce').fillna(0.0)
df['no']          = pd.to_numeric(df['no'], errors='coerce')
df['step_name_1'] = df['step_name_1'].astype(str).str.strip()
df['step_name_2'] = df['step_name_2'].astype(str).str.strip()

df['test_id'] = df['step_name_1'] + " / " + df['step_name_2']

print(f"Unique test_ids: {df['test_id'].nunique()}")
print(f"Sample test_ids: {df['test_id'].unique()[:5]}")

In [ ]:
unit_summary = df.groupby('unit_id').agg(
    overall_outcome    = ('unit_pass',    'first'),
    total_test_time    = ('step_time',    'sum'),
    num_steps_executed = ('test_id',      'count'),
    num_steps_failed   = ('result_binary', lambda x: (x == 0).sum())
).reset_index()

n_total = len(unit_summary)
n_pass  = unit_summary['overall_outcome'].sum()
n_fail  = (unit_summary['overall_outcome'] == 0).sum()
threshold     = round(n_fail / n_total * 1_000_000, 1)

print(f"\n{'='*50}")
print("EOL UNIT SUMMARY")
print(f"{'='*50}")
print(f"Total units:        {n_total}")
print(f"Passing units:      {n_pass}")
print(f"Failing units:      {n_fail}")
print(f"threshold (observed):     {threshold}")
print(f"Mean test time:     {unit_summary['total_test_time'].mean():.1f} s")
print(f"Min  test time:     {unit_summary['total_test_time'].min():.1f} s")
print(f"Max  test time:     {unit_summary['total_test_time'].max():.1f} s")

In [ ]:
all_failing = unit_summary[
    unit_summary['overall_outcome'] == 0
]['unit_id'].tolist()

genuine_fail_units = []
abort_only_units   = []

for unit_id in all_failing:
    unit_steps    = df[df['unit_id'] == unit_id]
    has_step_fail = (unit_steps['result'] == 'FAIL').any()
    if has_step_fail:
        genuine_fail_units.append(unit_id)
    else:
        abort_only_units.append(unit_id)

failing_units = genuine_fail_units
N_fail        = len(failing_units)

print(f"Total failing:    {len(all_failing)}")
print(f"Genuine FAIL:     {N_fail}   ← used for RQ1/RQ2/RQ3")
print(f"ABORT-only:       {len(abort_only_units)}  ← excluded")

In [ ]:
step_summary = df.groupby('test_id').agg(
    mean_cost_s      = ('step_time',     'mean'),
    total_executions = ('result_binary', 'count'),
    total_fails      = ('result_binary', lambda x: (x == 0).sum()),
    fail_rate_pct    = ('result_binary', lambda x: round((x == 0).mean() * 100, 4))
).reset_index().sort_values('total_fails', ascending=False)

print(f"\n{'='*50}")
print("EOL TEST STEP SUMMARY")
print(f"{'='*50}")
print(f"Total unique test steps:     {len(step_summary)}")
print(f"Steps with at least 1 FAIL:  {(step_summary['total_fails'] > 0).sum()}")
print(f"Steps with zero FAILs:       {(step_summary['total_fails'] == 0).sum()}")
print(f"\nTop 15 most failing steps:")
print(step_summary.head(15)[[
    'test_id', 'mean_cost_s', 'total_executions', 'total_fails', 'fail_rate_pct'
]].to_string(index=False))

In [ ]:
outcome_matrix = df.pivot_table(
    index='unit_id',
    columns='test_id',
    values='result_binary',
    aggfunc='first'
)

print(f"\n{'='*50}")
print("EOL OUTCOME MATRIX")
print(f"{'='*50}")
print(f"Shape: {outcome_matrix.shape}  (units x test steps)")
print(f"Missing entries: {outcome_matrix.isna().sum().sum()}")

In [ ]:
cost_vector = df.groupby('test_id')['step_time'].mean().round(4)
C_full_cost = cost_vector.sum()

print(f"Total full sequence cost (mean): {C_full_cost:.1f} seconds")
print(f"\nTop 10 most expensive steps:")
print(cost_vector.sort_values(ascending=False).head(10).to_string())

In [ ]:
df.to_csv("eol_cleaned.csv", index=False)
outcome_matrix.to_csv("eol_outcome_matrix.csv")
cost_vector.to_csv("eol_cost_vector.csv", header=True)
unit_summary.to_csv("eol_unit_summary.csv", index=False)
step_summary.to_csv("eol_step_summary.csv", index=False)

print(f"Saved 5 files:")
print(f"  eol_cleaned.csv")
print(f"  eol_outcome_matrix.csv")
print(f"  eol_cost_vector.csv")
print(f"  eol_unit_summary.csv")
print(f"  eol_step_summary.csv")

In [ ]:
outcome_matrix = pd.read_csv("eol_outcome_matrix.csv", index_col=0)
cost_vector    = pd.read_csv("eol_cost_vector.csv",    index_col=0, header=0)
cost_vector.columns = ['mean_cost_s']
unit_summary   = pd.read_csv("eol_unit_summary.csv")

df_check = pd.read_csv("eol_cleaned.csv", low_memory=False)

all_failing_saved  = unit_summary[unit_summary['overall_outcome'] == 0]['unit_id'].tolist()
genuine_fail_units = []
abort_only_units   = []

for unit_id in all_failing_saved:
    unit_steps    = df_check[df_check['unit_id'] == unit_id]
    has_step_fail = (unit_steps['result'] == 'FAIL').any()
    if has_step_fail:
        genuine_fail_units.append(unit_id)
    else:
        abort_only_units.append(unit_id)

failing_units = genuine_fail_units
N_fail        = len(failing_units)
C_full_cost   = cost_vector['mean_cost_s'].sum()

print(f"Total units:          {len(unit_summary)}")
print(f"Failing units |UF|:   {N_fail}")
print(f"Total test steps:     {outcome_matrix.shape[1]}")
print(f"Full sequence cost:   {C_full_cost:.2f} seconds")

In [ ]:
fail_matrix      = outcome_matrix.loc[outcome_matrix.index.isin(failing_units)].copy()
detection_matrix = (fail_matrix == 0).astype(float).fillna(0)

print(f"Detection matrix: {detection_matrix.shape}")
print(f"  rows = failing units,  cols = test steps")

In [ ]:
step_detection_count = detection_matrix.sum(axis=0)
diagnostic_steps     = step_detection_count[step_detection_count > 0].index.tolist()
non_diagnostic_steps = step_detection_count[step_detection_count == 0].index.tolist()

print(f"Diagnostic steps     (detect ≥1 failure): {len(diagnostic_steps)}")
print(f"Non-diagnostic steps (detect 0 failures): {len(non_diagnostic_steps)}")
print(f"\nTop 10 most diagnostic steps:")
print(step_detection_count.sort_values(ascending=False).head(10).to_string())

In [ ]:
costs = cost_vector.reindex(outcome_matrix.columns)['mean_cost_s'].fillna(0)

print(f"Full sequence cost: {C_full_cost:.2f} s")
print(f"Failing units:      {N_fail}")

In [ ]:
print(f"\n{'='*55}")
print("GREEDY SET COVER — selecting C* (RQ1)")
print(f"{'='*55}")

diag_detect = detection_matrix[diagnostic_steps].copy()
costs_diag  = cost_vector.reindex(diagnostic_steps)['mean_cost_s'].fillna(0)

uncovered    = set(failing_units)
selected     = []
trace        = []
already_used = set()
step_num     = 1

while uncovered:
    best_step, best_score, best_newly = None, -1, set()

    for step in diagnostic_steps:
        if step in already_used:
            continue
        newly_detected = set(
            diag_detect.index[
                (diag_detect[step] == 1) &
                (diag_detect.index.isin(uncovered))
            ].tolist()
        )
        if len(newly_detected) == 0:
            continue
        score = len(newly_detected) / max(costs_diag[step], 0.001)
        if score > best_score:
            best_score, best_step, best_newly = score, step, newly_detected

    if best_step is None:
        break

    already_used.add(best_step)
    uncovered -= best_newly
    selected.append(best_step)
    trace.append({
        'rank':                 step_num,
        'test_id':              best_step,
        'cost_s':               round(costs_diag[best_step], 4),
        'units_newly_detected': len(best_newly),
        'units_remaining':      len(uncovered),
        'cumulative_cost_s':    round(sum(costs_diag[s] for s in selected), 2)
    })
    print(f"{step_num:2d}. {best_step[:55]:<55} "
          f"detects {len(best_newly):3d} | remaining {len(uncovered):3d}")
    step_num += 1

In [ ]:
C_star_cost      = sum(costs_diag[s] for s in selected)
cost_saving_s    = C_full_cost - C_star_cost
cost_saving_pct  = cost_saving_s / C_full_cost * 100
steps_saving_pct = (1 - len(selected) / outcome_matrix.shape[1]) * 100
escaped          = len(uncovered)
escape_risk      = escaped / N_fail
escape_threshold = escape_risk * 1_000_000

print(f"\n{'='*55}")
print("RQ1 RESULTS — MINIMUM COST SUBSET C* (EOL)")
print(f"{'='*55}")
print(f"Steps in C*:               {len(selected)} / {outcome_matrix.shape[1]}")
print(f"Steps eliminated:          {outcome_matrix.shape[1]-len(selected)} ({steps_saving_pct:.1f}%)")
print(f"Cost of C* per unit:       {C_star_cost:.2f} seconds")
print(f"Cost of full sequence:     {C_full_cost:.2f} seconds")
print(f"Time saved per unit:       {cost_saving_s:.2f} seconds")
print(f"Cost saving (RQ1):         {cost_saving_pct:.2f}%")
print(f"Failing units detected:    {N_fail - escaped} / {N_fail}")
print(f"Escaped units:             {escaped}")
print(f"Escape risk:               {escape_risk:.6f}")
print(f"Escape threshold:          {escape_threshold:.1f}")

In [ ]:
non_diag_costs = cost_vector.reindex(non_diagnostic_steps)['mean_cost_s'].fillna(0)

non_diag_df = pd.DataFrame({
    'test_id': non_diagnostic_steps,
    'cost_s':  non_diag_costs.values
}).sort_values('cost_s', ascending=False)

print(f"\n{'='*55}")
print("NON-DIAGNOSTIC STEPS — zero defect detection value")
print(f"{'='*55}")
print(f"Count:                 {len(non_diag_df)}")
print(f"Total cost if skipped: {non_diag_df['cost_s'].sum():.2f} s/unit")
print(f"\nTop 10 most expensive non-diagnostic steps:")
print(non_diag_df.head(10).to_string(index=False))

In [ ]:
trace_df = pd.DataFrame(trace)
trace_df.to_csv("eol_rq1_c_star.csv", index=False)
non_diag_df.to_csv("eol_rq1_non_diagnostic.csv", index=False)

rq1_summary = {
    'c_star_steps':     len(selected),
    'c_star_cost_s':    round(C_star_cost, 2),
    'c_full_cost_s':    round(C_full_cost, 2),
    'cost_saving_s':    round(cost_saving_s, 2),
    'cost_saving_pct':  round(cost_saving_pct, 2),
    'steps_eliminated': outcome_matrix.shape[1] - len(selected),
    'steps_saving_pct': round(steps_saving_pct, 2),
    'escape_risk':      round(escape_risk, 6),
    'escape_threshold': round(escape_threshold, 1),
    'c_star_list':      str(selected)
}
pd.DataFrame([rq1_summary]).to_csv("eol_rq1_summary.csv", index=False)

print("Saved:")
print("  eol_rq1_c_star.csv         — selected steps with greedy trace")
print("  eol_rq1_non_diagnostic.csv — steps safe to eliminate")
print("  eol_rq1_summary.csv        — summary statistics")
print("\nComplete — ready for RQ2 notebook")

In [ ]:
# Verify C* catches all failures
missed = []
for unit_id in failing_units:
    if unit_id not in outcome_matrix.index:
        continue
    row = outcome_matrix.loc[unit_id]
    caught = any(step in row.index and row[step] == 0 for step in selected)
    if not caught:
        missed.append(unit_id)

print(f"C* steps:        {len(selected)}")
print(f"Failing units:   {N_fail}")
print(f"Caught by C*:    {N_fail - len(missed)}")
print(f"Missed by C*:    {len(missed)}")
print(f"Escape risk:     {len(missed)/N_fail:.6f}")
print(f"\nC* steps selected:")
for i, s in enumerate(selected, 1):
    print(f"  {i}. {s}")